# 프롬프트 엔지니어링(Prompt Engineering)

프롬프트 엔지니어링은 단순히 질문을 던지는 것을 넘어, 모델의 작동 원리와 '인컨텍스트 러닝(In-context Learning)' 능력을 활용해 모델의 출력을 제어하는 프로세스이다. 이는 모델의 파라미터(가중치)를 직접 수정하지 않고도 모델의 성능을 특정 태스크에 맞게 조정하는 방법론이다.

**프롬프트의 핵심 구성 요소:**

효과적인 프롬프트는 일반적으로 다음의 4가지 요소를 포함한다.

* **지시문 (Instruction):** 모델이 수행해야 할 구체적인 작업(예: 요약하라, 분류하라, 번역하라 등).
* **문맥 (Context):** 모델이 작업을 더 잘 수행하도록 돕는 배경 정보나 제약 조건.
* **입력 데이터 (Input Data):** 처리가 필요한 실제 데이터.
* **출력 지시자 (Output Indicator):** 결과물의 형식이나 스타일 지정(예: 표로 정리하라, JSON 포맷으로 출력하라 등).

**프롬프트 엔지니어링의 중요성:**

* **성능 최적화:** 같은 모델이라도 프롬프트에 따라 성능 차이가 극심하다. 잘 설계된 프롬프트는 더 작은 모델로도 큰 모델 수준의 결과를 낼 수 있게 한다.
* **비용 효율성:** 불필요한 토큰 사용을 줄이고, 파인튜닝(Fine-tuning)에 비해 적은 비용으로 도메인 특화 작업을 수행할 수 있다.
* **한계 극복:** 모델의 환각 현상을 줄이고 최신 정보를 반영(RAG와 결합 시)하도록 유도할 수 있다.

In [18]:
from dotenv import load_dotenv  # .env 환경변수 로드
from openai import OpenAI       # 클라이언트 객체 생성 클래스
import os                       # 환경변수 접근
import json                     

load_dotenv()   # .env 파일 읽어와 환경변수 등록
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')    # .env 파일의 OPENAI_API_KEY 값 가져옴
client = OpenAI()   # OPENAI_API_KEY가 환경변수에 등록되어 있으면 (api_key = ...) 생략 가능

In [ ]:
# Chat Completion API 호출
response= client.chat.completions.create(
    model = 'gpt-5.6-luna',
    messages = [
        # 시스템 프롬프트 : 모델의 페르소나 / 규칙 설정
        {
            'role' : 'system',  
            'content' : [
                {
                    'type' : 'text',
                    # 모델의 역할 / 출력 예시/ 출력 규칙
                    'text' : "기자들이 송고한 제목에서 맞춤법/문법/의미/어조등을 고려해 최상의 뉴스제목을 뽑아내는 20년 경력의 뉴스제목교정가이드다.\n\n## Instruction\n교정이 필요한 기사 제목을 입력받아, 맞춤법과 띄어쓰기 오류, 문법 오류를 지적하고 고친 제목을 제시하세요.  \n아래 단계로 진행합니다:  \n1. 입력된 기사 제목을 면밀히 분석하여 맞춤법 오류, 띄어쓰기 실수, 문법 오류 등 문제점을 찾아 지적 항목으로 정리합니다.  \n2. 문제점을 모두 고친 교정된 기사 제목을 결과로 제시합니다.  \n3. 교정이 필요한 부분과 수정결과를 교정이유항목에 작성해주세요.\n4. 기사 제목에 오류가 여러 개 있을 경우, 각 오류를 번호를 매겨 명확히 구분하여 지적합니다.\n5. 독자의 관심을 끌수 있도록 간결하면서도 임팩트 있는 표현을 사용하세요.\n6. 어조가 지나치게 감정적이거나 부정적이라면, 적절히 중립적 표현을 사용하세요.\n7. 비속어/욕설등이 포함되어 있다면 이를 제거하고, 의미가 전달될수 있는 적절한 표현으로 수정하세요. \n\n## Output Format\n- 원래제목: [송고한 기사제목]\n- 교정제목: [교정한 기사제목]\n- 교정 이유:\n  1. [교정한 부분과 이유]\n  2. [교정한 부분과 이유]\n\n## Examples\n<예시1>  \n입력: \"코로나19 백신접종율 높히기 위한 대안마련 필요하다\"  \n출력:  \n- 원래제목: [송고한 기사제목]\n- 교정제목: \"코로나19 백신 접종률 높이기 위한 대안 마련 시급\"\n- 교정 이유:\n   1. '접종율'은 '접종률'이 맞는 표기입니다.\n   2. '높히기'는 '높이기'로, 맞춤법 오류입니다.\n   3. '대안마련'은 붙여쓰지 않고 '대안 마련'으로 띄어 써야 맞습니다.\n   4. 간결한 어미수정\n\n<예시2>  \n입력: \"정부, 연금개혁 지방화 시대 맞춰 적극 나서야한다\"  \n출력:  \n- 원래제목: [송고한 기사제목]\n- 교정제목: \"정부, 연금개혁 지방화 시대 맞춰 적극 나서야\"\n- 교정 이유:\n  - 간결한 어미 수정\n"
                }
            ]
        },
        # 유저 프롬프트 : 사용자의 실제 입력 데이터
        {
            'role': 'user', 
            'content': [
                {
                    'type': 'text',
                    'text': '## Input Data\n입력: 오디세이 열풍, 과연 1000만영화가 될것인가?'
                }
            ]
        }
    ],
    response_format = {'type': 'text'},     # 응답 형식
    temperature = 1,                        # 창의성/ 다양성 (낮으면 결정론적, 일관적 / 높으면 창의적.)
    max_completion_tokens = 2048,           # 최대 출력 토큰
    top_p = 1,                              # 누적확률 p까지의 후보를 샘플링 (1은 전체사용)
    frequency_penalty = 0,                  # 동일 단어 반복시 감점(반복 억제)
    presence_penalty = 0,                   # 이미 등장한 단어는 감점(반복 억제)
    store = False
)

In [26]:
print(response.choices[0].message.content)

- 원래제목: 오디세이 열풍, 과연 1000만영화가 될것인가?
- 교정제목: **오디세이 열풍, 과연 1,000만 관객 돌파할까**
- 교정 이유:
  1. ‘1000만영화’는 ‘1,000만 관객’으로 풀어 써 의미를 명확히 했습니다.
  2. ‘될것인가’는 ‘될 것인가’로 띄어 써야 합니다.
  3. ‘될 것인가’를 ‘돌파할까’로 바꿔 제목을 간결하고 자연스럽게 다듬었습니다.
  4. 물음표는 제목의 간결성을 위해 생략했습니다.


In [ ]:
# 기사 제목 교정 API 호출 
def correct_headline(headline, /, *, model = 'gpt-5.6-luna', temperature= 1, top_p= 1, max_completion_tokens= 2048):
    
    response= client.chat.completions.create(
        model = model,
        messages = [
            # 시스템 프롬프트 : 모델의 페르소나 / 규칙 설정
            {
                'role' : 'system',  
                'content' : [
                    {
                        'type' : 'text',
                        # 모델의 역할 / 출력 예시/ 출력 규칙
                        'text' : "기자들이 송고한 제목에서 맞춤법/문법/의미/어조등을 고려해 최상의 뉴스제목을 뽑아내는 20년 경력의 뉴스제목교정가이드다.\n\n## Instruction\n교정이 필요한 기사 제목을 입력받아, 맞춤법과 띄어쓰기 오류, 문법 오류를 지적하고 고친 제목을 제시하세요.  \n아래 단계로 진행합니다:  \n1. 입력된 기사 제목을 면밀히 분석하여 맞춤법 오류, 띄어쓰기 실수, 문법 오류 등 문제점을 찾아 지적 항목으로 정리합니다.  \n2. 문제점을 모두 고친 교정된 기사 제목을 결과로 제시합니다.  \n3. 교정이 필요한 부분과 수정결과를 교정이유항목에 작성해주세요.\n4. 기사 제목에 오류가 여러 개 있을 경우, 각 오류를 번호를 매겨 명확히 구분하여 지적합니다.\n5. 독자의 관심을 끌수 있도록 간결하면서도 임팩트 있는 표현을 사용하세요.\n6. 어조가 지나치게 감정적이거나 부정적이라면, 적절히 중립적 표현을 사용하세요.\n7. 비속어/욕설등이 포함되어 있다면 이를 제거하고, 의미가 전달될수 있는 적절한 표현으로 수정하세요. \n\n## Output Format\n- 원래제목: [송고한 기사제목]\n- 교정제목: [교정한 기사제목]\n- 교정 이유:\n  1. [교정한 부분과 이유]\n  2. [교정한 부분과 이유]\n\n## Examples\n<예시1>  \n입력: \"코로나19 백신접종율 높히기 위한 대안마련 필요하다\"  \n출력:  \n- 원래제목: [송고한 기사제목]\n- 교정제목: \"코로나19 백신 접종률 높이기 위한 대안 마련 시급\"\n- 교정 이유:\n   1. '접종율'은 '접종률'이 맞는 표기입니다.\n   2. '높히기'는 '높이기'로, 맞춤법 오류입니다.\n   3. '대안마련'은 붙여쓰지 않고 '대안 마련'으로 띄어 써야 맞습니다.\n   4. 간결한 어미수정\n\n<예시2>  \n입력: \"정부, 연금개혁 지방화 시대 맞춰 적극 나서야한다\"  \n출력:  \n- 원래제목: [송고한 기사제목]\n- 교정제목: \"정부, 연금개혁 지방화 시대 맞춰 적극 나서야\"\n- 교정 이유:\n  - 간결한 어미 수정\n"
                    }
                ]
            },
            # 유저 프롬프트 : 사용자의 실제 입력 데이터
            {
                'role': 'user', 
                'content': [
                    {
                        'type': 'text',
                        'text': f'## Input Data\n입력: {headline}'
                    }
                ]
            }
        ],
        response_format = {'type': 'text'},               # 응답 형식
        temperature = temperature,                        # 창의성/ 다양성 (낮으면 결정론적, 일관적 / 높으면 창의적.)
        max_completion_tokens = max_completion_tokens,    # 최대 출력 토큰
        top_p = top_p,                          # 누적확률 p까지의 후보를 샘플링 (1은 전체사용)
        frequency_penalty = 0,                  # 동일 단어 반복시 감점(반복 억제)
        presence_penalty = 0,                   # 이미 등장한 단어는 감점(반복 억제)
        store = False                           # 응답을 서버에 저장/ 로깅 할지 여부
    )
    
    return response.choices[0].message.content

- 함수 인자 /, *
    - / : /의 왼쪽 매개변수는 위치인자방식으로 호출 강제화(headline은 무조건 첫번째 위치)
    - \* : \*의 오른쪽 매개변수는 키워드인자방식으로만 호출 강제화 (model = ..., temperature= ... 키워드 방식으로 사용)

In [30]:
print(correct_headline('민생지원금 뿌리다가 나라 파산'))

- 원래제목: [민생지원금 뿌리다가 나라 파산]
- 교정제목: [민생지원금 지급 확대에 국가 재정 악화 우려]
- 교정 이유:
  1. ‘뿌리다가’는 구어적이고 부정적인 표현이므로, 기사 제목에 맞게 ‘지급 확대’로 수정했습니다.
  2. ‘나라 파산’은 국가 상황을 단정적으로 표현한 부적절한 표현이므로, 객관적인 ‘국가 재정 악화 우려’로 바꿨습니다.
  3. 제목의 인과관계를 단정하지 않고 ‘우려’로 표현해 중립성과 정확성을 높였습니다.


In [32]:
headlines = [
    '두쫀쿠 다음 유행 상품은?',
    '가을야구 진출팀은?',
    'AI 채용시장'
]

for headline in headlines:
    output = correct_headline(headline)
    print(output + '\n')

- 원래제목: 두쫀쿠 다음 유행 상품은?
- 교정제목: 두쫀쿠 열풍 이을 다음 유행 상품은?
- 교정 이유:
  1. ‘두쫀쿠 다음’은 다소 단순하고 구어적인 표현이어서, ‘두쫀쿠 열풍 이을’로 바꿔 제목의 주목도를 높였습니다.
  2. ‘두쫀쿠’가 널리 알려진 상품명이 아니라면, 본문이나 부제에서 ‘두바이 초콜릿 쫀득 쿠키’ 등 정확한 명칭을 함께 제시하는 것이 좋습니다.

- 원래제목: 가을야구 진출팀은?
- 교정제목: 가을야구 진출 팀은?

- 교정 이유:
  1. ‘진출팀’은 ‘진출 팀’으로 띄어 쓰는 것이 적절합니다. ‘진출’과 ‘팀’은 각각 독립된 명사이므로 띄어 씁니다.

- 원래제목: AI 채용시장
- 교정제목: AI 채용시장
- 교정 이유:
  1. 맞춤법과 띄어쓰기상 오류가 없습니다. ‘AI 채용시장’은 AI를 활용한 채용시장이라는 의미로 자연스럽게 표현할 수 있습니다.



In [37]:
# 기사 제목 교정 API 호출 -> 결과를 json형식으로 받아 dict 형으로 반환하는 함수 
def correct_headline_json(headline, /, *, model = 'gpt-5.6-luna', temperature= 1, top_p= 1, max_completion_tokens= 2048):
    
    response= client.chat.completions.create(
        model = model,
        messages = [
            # 시스템 프롬프트 : 모델의 페르소나 / 규칙 설정
            {
                'role' : 'system',  
                'content' : [
                    {
                        'type' : 'text',
                        # 모델의 역할 / 출력 예시/ 출력 규칙
                        'text' : "기자들이 송고한 제목에서 맞춤법/문법/의미/어조등을 고려해 최상의 뉴스제목을 뽑아내는 20년 경력의 뉴스제목교정가이다.\n\n## Instruction\n교정이 필요한 기사 제목을 입력받아, 맞춤법과 띄어쓰기 오류, 문법 오류를 지적하고 고친 제목을 제시하세요.  \n아래 단계로 진행합니다:  \n1. 입력된 기사 제목을 면밀히 분석하여 맞춤법 오류, 띄어쓰기 실수, 문법 오류 등 문제점을 찾아 지적 항목으로 정리합니다.  \n2. 문제점을 모두 고친 교정된 기사 제목을 결과로 제시합니다.  \n3. 교정이 필요한 부분과 수정결과를 교정이유항목에 작성해주세요.\n4. 기사 제목에 오류가 여러 개 있을 경우, 각 오류를 번호를 매겨 명확히 구분하여 지적합니다.\n5. 독자의 관심을 끌수 있도록 간결하면서도 임팩트 있는 표현을 사용하세요.\n6. 어조가 지나치게 감정적이거나 부정적이라면, 적절히 중립적 표현을 사용하세요.\n7. 비속어/욕설등이 포함되어 있다면 이를 제거하고, 의미가 전달될수 있는 적절한 표현으로 수정하세요. \n\n## Output Format\n**반드시 json 객체 형식을 준수하세요.**\n\n{{\n  \"original_headline\": <송고한 기사제목>,\n  \"corrected_headline\": <교정한 기사제목>,\n  \"reasons\": [\n     <교정한 부분과 이유>,\n     <교정한 부분과 이유>,\n  ] \n}}\n\n## Examples\n<예시1>  \n입력: \"코로나19 백신접종율 높히기 위한 대안마련 필요하다\"  \n출력:  \n{{\n  \"original_headline\": \"코로나19 백신접종율 높히기 위한 대안마련 필요하다\",\n  \"corrected_headline\": \"코로나19 백신 접종률 높이기 위한 대안 마련 시급\",\n  \"reasons\": [\n    \"'접종율'은 표준어가 아니며 '접종률'이 올바른 표기이다\",\n    \"'높히기'는 맞춤법 오류로 '높이기'로 수정해야 한다\",\n    \"'대안마련'은 띄어 써야 하므로 '대안 마련'으로 수정하였다\",\n    \"기사 제목에 맞게 어미를 간결하게 다듬었다\"\n  ]\n}}\n\n<예시2>  \n입력: \"정부, 연금개혁 지방화 시대 맞춰 적극 나서야한다\"  \n출력:  \n{{\n  \"original_headline\": \"정부, 연금개혁 지방화 시대 맞춰 적극 나서야한다\",\n  \"corrected_headline\": \"정부, 연금개혁 지방화 시대 맞춰 적극 나서야\",\n  \"reasons\": [\n    \"기사 제목의 문체에 맞게 불필요한 어미를 제거해 간결하게 수정하였다\"\n  ]\n}}\n"
                    }
                ]
            },
            # 유저 프롬프트 : 사용자의 실제 입력 데이터
            {
                'role': 'user', 
                'content': [
                    {
                        'type': 'text',
                        'text': f'## Input Data\n입력: {headline}'
                    }
                ]
            }
        ],
        response_format = {'type': 'json_object'},               # 응답 형식
        temperature = temperature,                        # 창의성/ 다양성 (낮으면 결정론적, 일관적 / 높으면 창의적.)
        max_completion_tokens = max_completion_tokens,    # 최대 출력 토큰
        top_p = top_p,                          # 누적확률 p까지의 후보를 샘플링 (1은 전체사용)
        frequency_penalty = 0,                  # 동일 단어 반복시 감점(반복 억제)
        presence_penalty = 0,                   # 이미 등장한 단어는 감점(반복 억제)
        store = False                           # 응답을 서버에 저장/ 로깅 할지 여부
    )
    
    return  json.loads(response.choices[0].message.content)     # json -> python dict 파싱

In [40]:
print(correct_headline_json('심해지는 세대혐오 뭐가 문제일까')['corrected_headline'])

심해지는 세대 혐오, 무엇이 문제인가


In [42]:
# 입력 재료들을 이용해 조리 가능한 음식 2가지를 json형식으로 받아 dict 형으로 반환하는 함수 
def chef_json(user_input : str, /, *, model = 'gpt-5.6-luna', temperature= 1, top_p= 1, max_completion_tokens= 2048):
    
    response= client.chat.completions.create(
        model = model,
        messages = [
            # 시스템 프롬프트 : 모델의 페르소나 / 규칙 설정
            {
                'role' : 'system',  
                'content' : [
                    {
                        'type' : 'text',
                        # 모델의 역할 / 출력 예시/ 출력 규칙
                        'text' : "# Instruction\n사용자가 입력한 냉장고 내 재료 목록만을 사용하여 만들 수 있는 음식 2가지를 추천하세요.  \n반드시 입력된 재료만 활용하며, 기본양념(간장, 소금, 설탕, 설탕, 식초, 후추 등)은 언제든 사용할 수 있다고 가정하세요.  \n입력된 목록에 없는 재료(양념 제외)는 절대 사용하지 말고, 추가 재료 없이 조리 가능한 음식만 선정하십시오.  \n음식 종류는 반드시 서로 비슷하지 않은 2가지여야 하며, 각 음식에 대한 자세한 요리 레시피(조리 순서)를 단계별로 포함하세요.\n\n- 먼저, 입력 재료로 만들 수 있는 음식 종류를 논리적으로 검토한 뒤, 각 음식이 왜 가능한지 간단히 설명해 주세요.\n- reasoning(논리 및 설명)과 conclusion(최종 추천 결과)은 반드시 JSON의 개별 필드로 구분하여 제공하십시오.\n- reasoning이 반드시 먼저, conclusion이 반드시 마지막에 위치해야 합니다.\n- 결론(conclusion)에는 각 음식명과 단계별 레시피를 포함하세요.\n- 반드시 모든 답변을 한글로 작성하세요.\n\n# Steps\n\n1. 입력 재료만 활용 가능한 음식 2가지를 선정하고, 서로 비슷하지 않은지 확인하세요.\n2. reasoning(논리/설명) 필드에: \n    - 해당 재료로 어떤 음식이 가능한지, 그 이유를 간단히 단계별 논리로 설명하세요.\n3. conclusion(최종 추천) 필드에:\n    - 각 음식의 음식명\n    - 해당 음식의 구체적 요리 레시피(순서대로 단계를 나열, 최소 3단계 이상)\n    - 위 구조로 2가지만 반드시 작성하세요.\n\n# Output Format\n\n모든 답변은 아래 JSON 구조로 출력하세요.  \n- \"reasoning\": 각 음식이 왜 가능한지 단계별 논리와 검토(한글 서술, 리스트)\n- \"conclusion\": 음식명과 상세한 단계별 레시피(한글 서술, 리스트. 각 요소는 {\"food_name\": \"음식명\", \"recipe: [\"레시피1\", \"레시피2\"]} 형식)\n\n# Examples\n\n사용자 입력 예시:\n- 입력: 계란, 양파, 당근\n\n출력 예시(JSON):\n\n{\n  \"reasoning\": [\n    \"계란, 양파, 당근만 사용하여 만들 수 있는 요리를 검토합니다.\",\n    \"계란과 채소(양파, 당근)만으로 달걀전이 가능합니다. 채소를 잘게 썰어 계란과 섞어 부치면 완성할 수 있습니다.\",\n    \"계란찜 역시 이 재료로 만들 수 있습니다. 계란을 풀고 다진 채소를 섞은 후, 찜기를 사용해 익히면 완성됩니다.\"\n  ],\n  \"conclusion\": [\n    {\n      \"food_name\": \"달걀전\",\n      \"recipe\": [\n        \"1. 양파와 당근을 잘게 썰어줍니다.\",\n        \"2. 계란을 풀고 썰어둔 양파와 당근, 소금, 후추를 넣고 섞습니다.\",\n        \"3. 달궈진 팬에 기름을 두르고 반죽을 얇게 올린 후, 앞뒤로 노릇하게 부칩니다.\"\n      ]\n    },\n    {\n      \"food_name\": \"계란찜\",\n      \"recipe\": [\n        \"1. 계란을 볼에 넣고 곱게 풀어줍니다.\",\n        \"2. 다진 양파와 당근, 소금, 후추를 계란물에 넣고 섞습니다.\",\n        \"3. 뚝배기나 내열 용기에 재료를 옮겨 담고, 중탕 또는 전자레인지로 익혀 부드럽게 완성합니다.\"\n      ]\n    }\n  ]\n}\n\n(실제 예시는 입력 재료와 음식에 따라 달라지며, 각 음식의 레시피 단계는 3단계 이상, 충분히 구체적으로 작성하십시오.)\n\n# Notes\n\n- 반드시 입력 재료만 사용하고, 음식명 및 조리법 전부 한글로 기입하세요.\n- 각 추천 요리는 서로 다른 종류여야 하며, 각 음식마다 레시피 단계는 구체적이고 논리적으로 작성돼야 합니다.\n- reasoning(논리/설명) → conclusion(최종 추천 및 레시피) 순서는 꼭 지켜야 합니다.\n- 답변 형식은 반드시 JSON이어야 하며, 한글로만 작성하세요.\n\n[중요: 2가지 음식 추천, 상세 단계별 한글 레시피, 입력 재료만 허용, 양념장은 보유 가정, 항상 reasoning이 먼저, conclusion이 뒤, 반드시 JSON, 예시 구조 참고, 모든 답변은 한글로!]"
                    }
                ]
            },
            # 유저 프롬프트 : 사용자의 실제 입력 데이터
            {
                'role': 'user', 
                'content': [
                    {
                        'type': 'text',
                        'text': user_input
                    }
                ]
            }
        ],
        response_format = {'type': 'json_object'},               # 응답 형식
        temperature = temperature,                        # 창의성/ 다양성 (낮으면 결정론적, 일관적 / 높으면 창의적.)
        max_completion_tokens = max_completion_tokens,    # 최대 출력 토큰
        top_p = top_p,                          # 누적확률 p까지의 후보를 샘플링 (1은 전체사용)
        frequency_penalty = 0,                  # 동일 단어 반복시 감점(반복 억제)
        presence_penalty = 0,                   # 이미 등장한 단어는 감점(반복 억제)
        store = False                           # 응답을 서버에 저장/ 로깅 할지 여부
    )
    
    return  json.loads(response.choices[0].message.content)     # json -> python dict 파싱

In [43]:
chef_json('계란, 양파, 삼겹살')

{'reasoning': ['입력 재료는 계란, 양파, 삼겹살이며, 기본양념인 소금, 후추, 간장, 설탕 등은 사용할 수 있다고 가정합니다.',
  '삼겹살과 양파는 삼겹살에서 나온 기름으로 함께 볶을 수 있으므로 별도의 추가 재료 없이 양파 삼겹살 볶음을 만들 수 있습니다.',
  '계란과 양파는 계란물에 섞어 쪄낼 수 있으므로 계란 양파찜을 만들 수 있습니다.',
  '양파 삼겹살 볶음은 팬에 볶아 완성하는 고기 요리이고, 계란 양파찜은 쪄서 완성하는 계란 요리이므로 서로 다른 종류의 음식입니다.'],
 'conclusion': [{'food_name': '양파 삼겹살 볶음',
   'recipe': ['1. 삼겹살을 먹기 좋은 크기로 자르고, 양파는 껍질을 벗겨 굵게 채 썹니다.',
    '2. 달군 팬에 삼겹살을 올리고 중불에서 앞뒤로 익힙니다. 삼겹살에서 기름이 나오므로 별도의 식용유는 사용하지 않아도 됩니다.',
    '3. 삼겹살이 노릇하게 익고 기름이 충분히 나오면 채 썬 양파를 넣고 함께 볶습니다.',
    '4. 양파가 투명해지고 삼겹살이 완전히 익을 때까지 볶은 뒤, 간장과 설탕을 소량 넣어 간을 맞춥니다.',
    '5. 후추를 뿌리고 양념이 고기와 양파에 고르게 배도록 한 번 더 볶아 완성합니다.']},
  {'food_name': '계란 양파찜',
   'recipe': ['1. 양파를 잘게 다집니다.',
    '2. 계란을 그릇에 깨 넣고 소금과 후추를 넣어 충분히 풉니다.',
    '3. 다진 양파를 계란물에 넣고 고르게 섞습니다.',
    '4. 내열 용기에 계란물을 담고, 용기를 찜기에 넣어 중불에서 계란이 굳을 때까지 찝니다.',
    '5. 계란찜 가운데까지 단단하게 익었는지 확인한 뒤 꺼내 잠시 식혀 완성합니다.']}]}

In [44]:
output =chef_json('계란, 양파, 삼겹살, 찬밥')

for food in output['conclusion']:
    print(f'추천 음식: {food['food_name']}')
    print('레시피 :')
    for step in food['recipe']:
        print(' ', step)
    print()

추천 음식: 삼겹살 양파볶음
레시피 :
  1. 삼겹살을 먹기 좋은 크기로 자르고, 양파는 껍질을 벗긴 뒤 굵게 채 썹니다.
  2. 팬을 중간 불로 달군 뒤 식용유 없이 삼겹살을 올려 굽습니다.
  3. 삼겹살에서 지방이 충분히 나오고 겉면이 노릇해질 때까지 뒤집어가며 익힙니다.
  4. 채 썬 양파를 팬에 넣고 삼겹살과 함께 볶아 양파가 부드러워지게 합니다.
  5. 간장, 설탕, 후추를 넣고 재료에 양념이 골고루 배도록 볶습니다.
  6. 삼겹살이 완전히 익고 양파가 투명해지면 불을 끄고 그릇에 담습니다.

추천 음식: 계란 찬밥전
레시피 :
  1. 계란을 그릇에 깨 넣고 소금과 후추를 넣어 충분히 풉니다.
  2. 양파를 잘게 다져 계란물에 넣고, 찬밥도 함께 넣어 밥알이 뭉치지 않도록 골고루 섞습니다.
  3. 삼겹살에서 나온 기름이 남아 있다면 팬에 소량만 남기고, 중간 불로 팬을 달굽니다.
  4. 계란과 찬밥 혼합물을 팬에 한 국자씩 올려 둥글고 납작하게 모양을 잡습니다.
  5. 아랫면이 노릇하게 굳으면 뒤집어 반대쪽도 익힙니다.
  6. 계란이 완전히 익고 양파가 부드러워지면 접시에 담아 완성합니다.



In [49]:
# job Interview 준비 : 채용 공고/ 스펙을 입력받아, 면접 질문과 모범 답안을 Json(dict) 형태로 반환하는 함수
def job_interview_json(user_input : str, /, *, model = 'gpt-5.6-luna', temperature= 1, top_p= 1, max_completion_tokens= 2048):
    
    response= client.chat.completions.create(
        model = model,
        messages = [
            # 시스템 프롬프트 : 모델의 페르소나 / 규칙 설정
            {
                'role' : 'system',  
                'content' : [
                    {
                        'type' : 'text',
                        # 모델의 역할 / 출력 예시/ 출력 규칙
                        'text' : """
# Instruction
당신은 20년 경력의 ML/DL 엔지니어이고, 이번 신입개발자 채용의 면접관이다.
주어진 job posting(회사정보)를 바탕으로 신입개발자에 제공할 면접 질문과 모범 답변을 작성하세요.
크게 hard skill과 soft skill/leadership 질문을 구분하여 제시하십시오.

아래 지침을 반드시 준수하세요:

- 면접 질문 및 답변은 반드시 한글로, 그리고 신입 개발자에게 현실적으로 맞도록 작성합니다.
- hard skill(기술 질문)과 soft skill/leadership(소통, 태도, 리더십 등) 영역을 구분해 각각 최소 2개 이상의 예시를 만듭니다.
- 입력 정보가 부족한 경우, 통상적인 산업/포지션 상황에 근거해 논리적으로 추론해 주세요.
- 답변 전체의 output format은 반드시 아래 JSON 형태여야 하며,   hard_skill > soft_skill_leadership 순서로, 각 영역에 반드시 "질문"과 "모범답변"이 쌍으로 들어갑니다.
- 입력값 미제공 시 [회사정보], [스펙] 등의 placeholder를 사용하세요.

# Output Format

아래 JSON 포맷으로 여는 코드블럭 없이 반드시 출력합니다:

{
  "hard_skill": [
    {
      "question": "[hard skill(기술 중점) 면접 질문]",
      "answer": "[신입개발자 입장에서 모범답변]"
    },
    …
  ],
  "soft_skill_leadership": [
    {
      "question": "[soft skill/leadership(태도, 커뮤니케이션, 성장잠재력 등) 면접 질문]",
      "answer": "[신입개발자 입장에서 모범답변]"
    },
    …
  ]
}

- 각 영역별 면접 질문과 모범답변은 모두 한글로 작성합니다.
- 반드시 하드 스킬, 소프트 스킬/리더십 항목을 구분하여 각 2개 이상 작성합니다(총 4개 이상).
- 예시(Example)처럼 구조, 문장 길이, 구체성, 답변 스타일을 맞추되, 지원자는 신입개발자임을 꼭 반영하세요.

# 예시(Example)

Input:
회사정보: [AI 기업, 제조공정 데이터 분석 솔루션 개발]
스펙: [전자공학 전공, 신입 개발자, Python/Java 가능, 인턴 경험 있음]

Output:

{{
  "hard_skill": [
    {
      "question": "Python 언어를 사용하여 데이터 전처리 경험이 있나요? 예시를 들어 설명해 주세요.",
      "answer": "대학 시절 프로젝트에서 Pandas를 이용해 결측치 처리와 이상치 제거를 해본 경험이 있습니다. 데이터 정제의 중요성을 실제로 체감할 수 있었습니다."
    },
    {
      "question": "제조 데이터와 같이 구조적인 데이터를 다룰 때 주의해야 하는 점은 무엇이라고 생각하나요?",
      "answer": "데이터의 정확한 구조 파악과, 이상치 및 에러를 사전에 점검하는 것이 중요하다고 생각합니다."
    }
  ],
  "soft_skill_leadership": [
    {
      "question": "팀 프로젝트에서 갈등이 있을 때 어떻게 해결해보셨나요?",
      "answer": "인턴 경험 중 팀원과 의견이 달랐을 때 서로의 입장을 경청한 뒤 중간 지점을 찾아 협력한 경험이 있습니다."
    },
    {
      "question": "빠르게 변화하는 환경에서 새로운 기술을 습득했던 경험이 있나요?",
      "answer": "새로운 도구가 필요했던 프로젝트에서 적극적으로 온라인 자료를 찾아 공부하며 습득하였습니다."
    }
  ]
}}

# Notes
- 지원자는 신입개발자, 면접관 페르소나는 20년차 ml/dl 엔지니어임을 모든 답변에 일관되게 반영하세요.
- 모든 콘텍스트와 답변, 지시문, reasoning, 사례, placeholder, Q&A 등은 전체 한글로 작성합니다.
- 포맷(최상단 하드스킬 > 소프트스킬/리더십)과 다국어 혼용 불허, 신입 시점 미반영 등은 모두 미승인 처리 대상입니다.

**Objective Reminder:**
1. '20년차 ml/dl engineer' 페르소나로서, 신입 개발자 대상 면접 Q&A를 회사/직무 정보로부터  hard/soft skill/leaderhip별 제시
2. JSON 아웃풋 포맷만 사용 (코드블럭 넣지 않음)
"""
                        
                    }
                ]
            },
            # 유저 프롬프트 : 사용자의 실제 입력 데이터
            {
                'role': 'user', 
                'content': [
                    {
                        'type': 'text',
                        'text': user_input
                    }
                ]
            }
        ],
        response_format = {'type': 'json_object'},               # 응답 형식
        temperature = temperature,                        # 창의성/ 다양성 (낮으면 결정론적, 일관적 / 높으면 창의적.)
        max_completion_tokens = max_completion_tokens,    # 최대 출력 토큰
        top_p = top_p,                          # 누적확률 p까지의 후보를 샘플링 (1은 전체사용)
        frequency_penalty = 0,                  # 동일 단어 반복시 감점(반복 억제)
        presence_penalty = 0,                   # 이미 등장한 단어는 감점(반복 억제)
        store = False                           # 응답을 서버에 저장/ 로깅 할지 여부
    )
    
    return  json.loads(response.choices[0].message.content)     # json -> python dict 파싱

In [50]:
output = job_interview_json(
    """
    ## Job Descriptions

주요 업무 내용 안내

### 코딕스 개발팀 소개말

코딕스 개발팀은 콘텐츠 플랫폼 테스트북을 중심으로 일하고 있습니다. 사용자들이 더 편하게 서비스를 이용하도록 돕고있어요. 테스트북 뿐만 아니라 웹을 기반으로 한 다양한 서비스를 개발합니다.
기존에 사용하던 도구 이외에 새로운 Framework나 기술에도 관심이 많으며 적극적으로 검토하고 도입하기 위해 노력해요. 물론 개인의 역량과 커리어 증진에도 힘쓰고 있습니다.

---

## AI 개발자 주요 업무 내용 안내

### AI 개발자 (AI Developer)

AI 기술을 이용한 교육용 애플리케이션 프로젝트를 개발(ML/DL/Generative AI)하고,
Python 기반 API 서버 개발 및 Kubernetes 기반 배포 환경 구성합니다.

---

### 기술 및 자격 요건

* Python 프로그래밍 (FastAPI, Django, Flask 등) 경험이 있는 사람
* NLP 프로젝트 또는 LLM + RAG + VectorDB 기반 개발 경험이 있는 사람
* RESTful API 설계 및 Kubernetes 기반 배포 경험이 있는 사람
* 업무 커뮤니케이션에 대한 소통 능력이 뛰어난 분

---

### 우대 사항

* 바이브 코딩 경험을 보유하신 분
* 생성형 AI 관련 프로젝트 경험이 있는 사람
* 각종 협업 도구에 익숙하고 새로운 사용에 적극적인 사람
    """
    
)

In [51]:
output

{'hard_skill': [{'question': '교육용 AI 애플리케이션에서 파이썬 기반 API 서버를 개발한다면, 어떤 프레임워크를 선택하고 어떻게 설계하겠습니까?',
   'answer': '신입으로서 먼저 팀의 기존 기술과 서비스 요구사항을 확인한 뒤, 비동기 처리와 자동 문서화가 중요한 서비스라면 패스트에이피아이를 우선 검토하겠습니다. 요청과 응답 형식을 명확히 정의하고, 라우터·서비스·데이터 접근 계층을 분리해 유지보수성을 높이겠습니다. 또한 입력값 검증, 예외 처리, 인증과 권한 확인을 적용하고, 핵심 API는 테스트 코드로 동작을 검증하겠습니다.'},
  {'question': '대규모 언어 모델과 검색 증강 생성 구조를 교육용 서비스에 적용한다면, 전체 처리 과정을 설명해 주세요.',
   'answer': '먼저 교재나 학습 자료를 수집하고 문서 형식과 불필요한 내용을 정리합니다. 이후 문서를 적절한 크기로 나눈 뒤 임베딩 벡터로 변환해 벡터 데이터베이스에 저장합니다. 사용자의 질문이 들어오면 질문도 벡터로 변환해 관련 문서를 검색하고, 검색 결과를 대규모 언어 모델의 입력에 함께 전달해 답변을 생성합니다. 답변의 정확성을 높이기 위해 검색 문서의 출처를 함께 제공하고, 관련 없는 질문이나 근거가 부족한 경우에는 모른다고 답하도록 설계하겠습니다.'},
  {'question': '검색 증강 생성 시스템에서 답변 품질이 낮을 때 어떤 순서로 원인을 분석하겠습니까?',
   'answer': '먼저 실제 질문과 기대 답변을 기준으로 문제를 재현하겠습니다. 그다음 문서 수집 과정에서 내용이 누락되었는지, 문서 분할 크기가 적절한지, 임베딩 모델과 검색 결과가 질문과 관련 있는지 확인하겠습니다. 검색 결과는 적절하지만 답변이 잘못된다면 프롬프트와 모델의 응답 방식을 점검하겠습니다. 또한 검색된 문서의 관련도, 정답 포함 여부, 답변의 정확성과 근거 제시 여부를 별도로 평가해 어느 단계에서 문제가 발생하는지 구분하겠습니다.'},
  {'questi

In [52]:
for category, qa_list in output.items():
    print(f'[{category}]')
    
    for qa in qa_list:
        question = qa.get('question','').strip()
        answer = qa.get('answer','').strip()
        
        print(f'Q : {question}')
        print(f'A : {answer}')
        
    print()

[hard_skill]
Q : 교육용 AI 애플리케이션에서 파이썬 기반 API 서버를 개발한다면, 어떤 프레임워크를 선택하고 어떻게 설계하겠습니까?
A : 신입으로서 먼저 팀의 기존 기술과 서비스 요구사항을 확인한 뒤, 비동기 처리와 자동 문서화가 중요한 서비스라면 패스트에이피아이를 우선 검토하겠습니다. 요청과 응답 형식을 명확히 정의하고, 라우터·서비스·데이터 접근 계층을 분리해 유지보수성을 높이겠습니다. 또한 입력값 검증, 예외 처리, 인증과 권한 확인을 적용하고, 핵심 API는 테스트 코드로 동작을 검증하겠습니다.
Q : 대규모 언어 모델과 검색 증강 생성 구조를 교육용 서비스에 적용한다면, 전체 처리 과정을 설명해 주세요.
A : 먼저 교재나 학습 자료를 수집하고 문서 형식과 불필요한 내용을 정리합니다. 이후 문서를 적절한 크기로 나눈 뒤 임베딩 벡터로 변환해 벡터 데이터베이스에 저장합니다. 사용자의 질문이 들어오면 질문도 벡터로 변환해 관련 문서를 검색하고, 검색 결과를 대규모 언어 모델의 입력에 함께 전달해 답변을 생성합니다. 답변의 정확성을 높이기 위해 검색 문서의 출처를 함께 제공하고, 관련 없는 질문이나 근거가 부족한 경우에는 모른다고 답하도록 설계하겠습니다.
Q : 검색 증강 생성 시스템에서 답변 품질이 낮을 때 어떤 순서로 원인을 분석하겠습니까?
A : 먼저 실제 질문과 기대 답변을 기준으로 문제를 재현하겠습니다. 그다음 문서 수집 과정에서 내용이 누락되었는지, 문서 분할 크기가 적절한지, 임베딩 모델과 검색 결과가 질문과 관련 있는지 확인하겠습니다. 검색 결과는 적절하지만 답변이 잘못된다면 프롬프트와 모델의 응답 방식을 점검하겠습니다. 또한 검색된 문서의 관련도, 정답 포함 여부, 답변의 정확성과 근거 제시 여부를 별도로 평가해 어느 단계에서 문제가 발생하는지 구분하겠습니다.
Q : 파이썬으로 개발한 AI API를 쿠버네티스 환경에 배포할 때 기본적으로 고려할 사항은 무엇인가요?
A : 애플리케이션을 컨테이너 이미지로 만들고, 환경별 설정

In [ ]:
answer = []
n = 10
while n >=1 :
    answer.append(int(n))
    if n == 1:
            break
    if n % 2 ==0:
        n = n / 2
    else:
        n = 3*n +1


10
5.0
16.0
8.0
4.0
2.0
1.0


[10, 5, 16, 8, 4, 2, 1]